# Jyupiter Training Notebook

This notebook demonstrates how to train the `SvaraTransformer` model in the Raaga.AI repository using the existing training pipeline. It loads the dataset, builds the model, trains it, evaluates validation loss, and saves a checkpoint.

In [ ]:
import math
import time
import sys
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Locate the repository root by walking upward until the `src` folder is found
repo_root = Path.cwd()
while not (repo_root / 'src').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / 'src').exists():
    raise RuntimeError(
        f'Could not find repository root with a src directory from cwd={Path.cwd()}'
    )

sys.path.insert(0, str(repo_root))

from src.ml.data.dataset import SvaraDataset
from src.ml.model import SvaraTransformer
from src.ml.tokenizer import SvaraTokenizer

pretrained_checkpoint_path = repo_root / 'data' / 'models' / 'svara_transformer.pt'
notebook_checkpoint_path = repo_root / 'data' / 'models' / 'svara_transformer_notebook.pt'

print('Repository root:', repo_root)
print('Pretrained checkpoint:', pretrained_checkpoint_path)
print('Notebook checkpoint:', notebook_checkpoint_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)


: 

## Load Dataset

The Raaga.AI training data is expected at `data/training/svara_sequences.jsonl`. This notebook loads that file and inspects the dataset size.

In [ ]:
data_path = repo_root / 'data' / 'training' / 'svara_sequences.jsonl'
print('Training data path:', data_path)
print('Exists:', data_path.exists())

if not data_path.exists():
    raise FileNotFoundError('Training data not found. Run scripts/build_dataset.py first.')

tokenizer = SvaraTokenizer()
dataset = SvaraDataset(str(data_path), tokenizer, max_length=256)
print('Dataset size:', len(dataset))
print('Vocabulary size:', tokenizer.vocab_size)

sample = dataset[0] if len(dataset) > 0 else None
print('Sample keys:', list(sample.keys()) if sample else None)
if sample:
    print('Input IDs length:', sample['input_ids'].shape)
    print('Labels length:', sample['labels'].shape)


## Preprocess Data

Create train/validation splits and PyTorch data loaders for batch training. This step uses the same sequence format and tokenizer as the repo's training script.

In [ ]:
batch_size = 32
val_split = 0.1
max_seq_len = 256

val_size = max(1, int(len(dataset) * val_split))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)

print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Batch size:', batch_size)


## Define Model Architecture

Instantiate the `SvaraTransformer` from `src/ml/model.py` with the same architecture used by the existing training script.

In [ ]:
model = SvaraTransformer(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=max_seq_len,
    dropout=0.1,
).to(device)


def count_parameters(module: torch.nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

print('Model parameter count:', count_parameters(model))
print(model)


## Compile Model

Set up optimizer, scheduler, and training hyperparameters. This notebook mirrors the repo's `scripts/train_model.py` defaults.

If a pretrained checkpoint exists, it is loaded and training is skipped; otherwise the notebook will train for the full number of epochs on the available dataset.


In [ ]:
learning_rate = 3e-4
grad_clip = 1.0
full_epochs = 50

# If a pretrained checkpoint exists, use it and skip full training.
# Otherwise, train from scratch on the available dataset.
if notebook_checkpoint_path.exists():
    checkpoint_to_load = notebook_checkpoint_path
elif pretrained_checkpoint_path.exists():
    checkpoint_to_load = pretrained_checkpoint_path
else:
    checkpoint_to_load = None

if checkpoint_to_load is not None:
    epochs = 0
    print('Loading pretrained checkpoint, skipping training:', checkpoint_to_load)
    checkpoint = torch.load(checkpoint_to_load, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded pretrained model weights from: {checkpoint_to_load}')
else:
    epochs = full_epochs
    print('No pretrained checkpoint found; training for full epochs:', epochs)

optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=learning_rate * 0.1)

print('Learning rate:', learning_rate)
print('Epochs:', epochs)
print('Checkpoint to load:', checkpoint_to_load)


## Train Model

Run the training loop. The notebook logs training and validation loss per epoch and saves the best checkpoint.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    steps = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        output = model(input_ids, labels=labels)
        loss = output['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(steps, 1)


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    steps = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            output = model(input_ids, labels=labels)
            total_loss += output['loss'].item()
            steps += 1
    return total_loss / max(steps, 1)

output_dir = repo_root / 'data' / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = notebook_checkpoint_path

best_val_loss = float('inf')
patience = 10
patience_counter = 0
start_time = time.time()

if epochs == 0:
    print('Pretrained checkpoint loaded; skipping training loop.')
else:
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate(model, val_loader, device)
        scheduler.step()
        elapsed = time.time() - start_time
        print(f'Epoch {epoch}/{epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - lr: {scheduler.get_last_lr()[0]:.2e} - elapsed: {elapsed:.0f}s')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': {
                    'vocab_size': tokenizer.vocab_size,
                    'embed_dim': 128,
                    'num_heads': 4,
                    'num_layers': 4,
                    'max_seq_len': max_seq_len,
                    'dropout': 0.0,
                },
                'tokenizer_vocab': tokenizer.token_to_id,
                'epoch': epoch,
                'val_loss': best_val_loss,
            }, checkpoint_path)
            print(f'  → Saved best model to {checkpoint_path}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping after {patience} epochs without improvement.')
                break

    print('Training complete. Best validation loss:', best_val_loss)


## Evaluate Model

Run a final evaluation on the held-out validation split to confirm the checkpoint quality.

In [ ]:
final_val_loss = evaluate(model, val_loader, device)
print('Final validation loss:', final_val_loss)


## Save Trained Model

The notebook saves the trained checkpoint and generates a sample melody using the same tokenizer.

In [ ]:
print('Checkpoint saved at:', checkpoint_path)

model.eval()
prefix_tokens = tokenizer.encode_conditioning('shankarabharanam', 'adi', 'pallavi', 'madhyama')
prefix_tensor = torch.tensor([prefix_tokens], device=device)
generated = model.generate(prefix_tensor, max_new_tokens=32, temperature=0.8, top_k=15)
decoded = tokenizer.decode(generated[0].cpu().tolist())
print('Generated sequence:')
print(' '.join(decoded))
